# Tarea 1.2 · Matriz de decisión tecnológica

**Curso:** Herramientas de Big Data · Saber 2: Procesamiento distribuido
**Proyecto:** Emergencias de la Línea 123 de Bogotá (julio, agosto y septiembre de 2026)
**Actividad:** Actividad 1 · Fase 1 · Investigación

**Objetivos**
- Definir criterios y pesos: rendimiento 30 %, facilidad de uso 25 %, escalabilidad 20 %, recursos 15 %, curva de aprendizaje 10 %.
- Puntuar de 1 a 5 cada herramienta a partir de las mediciones del benchmark (tarea 1.1).
- Calcular el puntaje ponderado Σ(criterio × peso) y validarlo con el caso de uso.

**Prerrequisitos**
- `01_T1.1_Benchmark.ipynb` ejecutado (mediciones en `BigData_U2/resultados/mediciones/`).

**Tiempo estimado de ejecución:** ~1 minuto (Colab gratuito, solo CPU)

**Dataset:** «Llamadas de urgencias y emergencias que ingresan a través de la línea 123», Secretaría Distrital de
Salud, portal Datos Abiertos Bogotá (CC BY 4.0). Tres CSV mensuales (27 434 registros, separador `;`).

## 1. Lectura de las mediciones del benchmark

Solo se leen las mediciones guardadas por el notebook 01; aquí no se vuelve a medir.

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

MEDICIONES = '/content/drive/MyDrive/BigData_U2/resultados/mediciones'
bench = pd.read_csv(f'{MEDICIONES}/benchmark.csv')
loc = pd.read_csv(f'{MEDICIONES}/loc_benchmark.csv', index_col=0)
medidas = bench[bench['ronda'] > 0]
por_ronda = medidas.groupby(['herramienta', 'ronda']).agg(
    tiempo_total_s=('segundos', 'sum'), mem_pct_pico=('mem_pct_pico', 'max'), mem_pct_inicio=('mem_pct_antes', 'first'))
por_ronda['mem_incremento_pp'] = por_ronda['mem_pct_pico'] - por_ronda['mem_pct_inicio']
HERRAMIENTAS = ['pandas', 'polars', 'spark']
insumos = por_ronda.groupby('herramienta')[['tiempo_total_s', 'mem_incremento_pp']].mean().loc[HERRAMIENTAS]
insumos['lineas_codigo'] = loc.loc['TOTAL', HERRAMIENTAS].astype(int)
insumos.round(3)

## 2. Reglas de puntuación (1 = peor, 5 = mejor)

| Criterio | Peso | Cómo se puntúa |
|---|---:|---|
| Rendimiento | 30 % | Cuantitativo: tiempo total del benchmark. |
| Facilidad de uso | 25 % | Cuantitativo (líneas de código) con un ajuste cualitativo de −1 a +1. |
| Escalabilidad | 20 % | Cualitativo: 5 = multinodo nativo; 3 = un nodo multihilo; 2 = un nodo en memoria. |
| Recursos | 15 % | Cuantitativo: incremento pico de memoria del sistema (puntos porcentuales). |
| Curva de aprendizaje | 10 % | Cualitativo: 5 = API conocida; 4 = API nueva similar; 2 = conceptos de motor distribuido. |

Para los criterios cuantitativos se calcula `r = valor / mejor valor` y se asigna: `r ≤ 1,25 → 5`, `r ≤ 2 → 4`,
`r ≤ 4 → 3`, `r ≤ 8 → 2` y `r > 8 → 1`. En memoria, los incrementos menores de 0,1 puntos se toman como 0,1 para que
el cociente tenga sentido.

In [ ]:
def puntaje(valores):
    r = valores / valores.min()
    return r.apply(lambda x: 5 if x <= 1.25 else 4 if x <= 2 else 3 if x <= 4 else 2 if x <= 8 else 1)


AJUSTE_FACILIDAD = pd.Series({'pandas': 0, 'polars': 0, 'spark': -1})   # PySpark: SparkSession, JVM, evaluación perezosa
ESCALABILIDAD = pd.Series({'pandas': 2, 'polars': 3, 'spark': 5})
CURVA = pd.Series({'pandas': 5, 'polars': 4, 'spark': 2})
PESOS = pd.Series({'Rendimiento': 0.30, 'Facilidad de uso': 0.25, 'Escalabilidad': 0.20, 'Recursos': 0.15,
                   'Curva de aprendizaje': 0.10})

matriz = pd.DataFrame({
    'Rendimiento': puntaje(insumos['tiempo_total_s']),
    'Facilidad de uso': (puntaje(insumos['lineas_codigo']) + AJUSTE_FACILIDAD).clip(1, 5),
    'Escalabilidad': ESCALABILIDAD,
    'Recursos': puntaje(insumos['mem_incremento_pp'].clip(lower=0.1)),
    'Curva de aprendizaje': CURVA,
}).T[HERRAMIENTAS]
matriz.insert(0, 'Peso', PESOS)
matriz

## 3. Puntaje ponderado

In [ ]:
ponderado = matriz[HERRAMIENTAS].mul(matriz['Peso'], axis=0).sum().round(2)
resultado = pd.DataFrame({'Puntaje ponderado': ponderado,
                          'Posición': ponderado.rank(ascending=False, method='min').astype(int)})
print(f"Suma de pesos: {matriz['Peso'].sum():.0%}")
display(resultado.sort_values('Posición'))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh([h.capitalize() for h in HERRAMIENTAS], ponderado[HERRAMIENTAS], color=['#4C78A8', '#54A24B', '#F58518'])
for i, v in enumerate(ponderado[HERRAMIENTAS]):
    ax.text(v + 0.03, i, f'{v:.2f}', va='center')
ax.set_xlim(0, 5)
ax.set_xlabel('Puntaje ponderado (1–5)')
ax.set_title('Matriz de decisión: puntaje ponderado')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Validación con el caso de uso y restricciones de Colab

- **Caso de uso:** consolidar y validar mes a mes las llamadas de la Línea 123 (≈9 000 registros por mes) y dejar
  datos limpios en Parquet para análisis por localidad y tipo de incidente.
- **Restricciones de Colab gratuito:** ≈12 GB de RAM, 2 vCPU y sesiones de hasta 12 h. Con este volumen cualquier
  herramienta cabe en memoria; la diferencia aparece cuando el histórico crece (tarea 2.4).
- **Decisión:** la matriz muestra qué herramienta conviene con **este** volumen. PySpark es la herramienta
  **obligatoria** de la actividad y se justifica por su escalabilidad horizontal; si otra gana aquí, se reporta tal cual.

In [ ]:
matriz_json = {'pesos': PESOS.to_dict(),
               'puntajes': {c: {h: int(matriz.loc[c, h]) for h in HERRAMIENTAS} for c in PESOS.index},
               'ponderado': ponderado.to_dict(), 'insumos': insumos.round(4).to_dict()}
with open(f'{MEDICIONES}/matriz_decision.json', 'w', encoding='utf-8') as f:
    json.dump(matriz_json, f, indent=2, ensure_ascii=False)
mejor = ponderado.idxmax()
print(f"Mayor puntaje con este volumen: {mejor.capitalize()} ({ponderado[mejor]:.2f}). "
      f"PySpark: {ponderado['spark']:.2f}. Matriz guardada en {MEDICIONES}/matriz_decision.json")